In [ ]:
#LOSS

#TODO imposta i weights della DICE e della BCE
w_bce=1
w_dice=1
#L-SUP
class L_supBCEDice(nn.Module):
    def __init__(self, smooth=1.0):
        super(L_supBCEDice, self).__init__()
        self.smooth=smooth
        self.bce = nn.BCEWithLogitsLoss(reduction='none')  # per applicare pesi a livello di batch

    def dice_loss(self, logits, targets):
        probs=torch.sigmoid(logits)
        probs_flat=probs.view(probs.size(0),-1) #appiattisce ogni immagine in un vettore, calcolando la Dice per immagine (batch di dimensione B)
        targets_flat=targets.view(targets.size(0),-1) #same appiattimento
        intersection=(probs_flat*targets_flat).sum(dim=1) #somma dei prodotti p*y per ogni pixel (overlap predizione-ground truth)
        union=probs_flat.sum(dim=1)+targets_flat.sum(dim=1) #somma prediction+real (per il denominatore dice)
        dice=1-(2*intersection+self.smooth)/(union+self.smooth) #loss per immagine
        return dice

    def forward(self, logits, targets, is_positive):

        """
        logits: (B, 1, H, W)
        targets: (B, 1, H, W) o (B, H, W)
        is_positive: tensor bool/int di shape (B,) -> True se campione positivo, False se negativo
        """
        if targets.dim()==3: #se le immagini target hanno un canale in meno lo aggiungo per renderle compatibili con i logits
          targets=targets.unsqueeze(1).float()
        else:
          targets=targets.float()

        #converte in float perchè la loss funziona con float e non con long

        bce_per_pixel=self.bce(logits,targets) #(B,1,H,W)
        bce_per_img=bce_per_pixel.view(bce_per_pixel.size(0),-1).mean(dim=1) #(B,)
        dice_per_img=self.dice_loss(logits,targets)

        w_bce = torch.ones_like(bce_per_img)
        w_dice = is_positive.float()

        loss_per_img=w_bce*bce_per_img+w_dice*dice_per_img #combino le 2 loss con DICE spenta sui negativi
        loss=loss_per_img.mean()

        return loss